In [1]:
import sys

model_path = '/Users/rdobrien/Library/CloudStorage/OneDrive-NASA/skysurf_testing/Zodiacal_Light_Model_Python/'
if model_path not in sys.path:
    sys.path.append(model_path)
import zodisurf

from astropy.io import fits
import astropy.units as u
from astropy.coordinates import SkyCoord, get_body_barycentric, CartesianRepresentation
from astropy.time import Time

In [2]:
hdu = fits.open('/Users/rdobrien/Library/CloudStorage/OneDrive-NASA/jwst_backgrounds/scripts/query_jwst_data/downloads/prism_observations/2022-08-30/VID01191011001/jw01191011001_02101_00002_nrs1_uncal.fits')

In [3]:
prim_hdr = hdu[0].header
sci_hdr = hdu[1].header
targ_ra = sci_hdr['RA_REF'] * u.deg
targ_dec = sci_hdr['DEC_REF'] * u.deg
date_obs = prim_hdr['DATE-OBS']

In [4]:
# 1. Target coordinates from the header
target = SkyCoord(ra=targ_ra, dec=targ_dec, frame='icrs')

# 2. Convert JWST barycentric coordinates from km to AU directly during initialization
jwst_x = 138167444.05045882
jwst_y = -56061304.01820322
jwst_z = -24291033.467842504
jwst_cart_au = CartesianRepresentation(
    (jwst_x * u.km).to(u.AU),
    (jwst_y * u.km).to(u.AU),
    (jwst_z * u.km).to(u.AU)
)

# 3. Observation time
obs_time = Time(date_obs, format='isot', scale='utc')

# 4. Get the Sun's barycentric position at the observation time (returns in AU)
sun_bary = get_body_barycentric('sun', obs_time)

sun_jwst_cart = sun_bary - jwst_cart_au
sun_jwst_coord = SkyCoord(sun_jwst_cart, frame='icrs')

solar_elongation = target.separation(sun_jwst_coord)
print(f"JWST Solar Elongation: {solar_elongation.deg:.4f} degrees")

JWST Solar Elongation: 88.0927 degrees


In [5]:
import numpy as np
import astropy.units as u

jwst_icrs = SkyCoord(jwst_cart_au, frame='icrs')

jwst_ecl = jwst_icrs.transform_to('barycentrictrueecliptic')

jwst_x_ecl = jwst_ecl.cartesian.x
jwst_y_ecl = jwst_ecl.cartesian.y
jwst_z_ecl = jwst_ecl.cartesian.z

lon_rad = target.transform_to('barycentrictrueecliptic').lon.rad
lat_rad = target.transform_to('barycentrictrueecliptic').lat.rad

obs_Dis = np.sqrt(jwst_x_ecl**2 + jwst_y_ecl**2 + jwst_z_ecl**2)

dot_product = (
    -jwst_x_ecl * np.cos(lat_rad) * np.cos(lon_rad) +
    -jwst_y_ecl * np.cos(lat_rad) * np.sin(lon_rad) +
    -jwst_z_ecl * np.sin(lat_rad)
)

obs_SolElong = np.arccos(dot_product / obs_Dis)
print('obs_SolElong (deg):', np.degrees(obs_SolElong))

obs_SolElong (deg): 88.23371930035276 deg


In [7]:
day = 242
lon_deg = target.transform_to('barycentrictrueecliptic').lon.deg
lat_deg = target.transform_to('barycentrictrueecliptic').lat.deg
zodisurf.get_zmod(1.0,'skysurf', day, lon_deg, lat_deg, obsxyz=(jwst_x, jwst_y, jwst_z))
zodisurf.get_zmod(1.0,'skysurf', day, lon_deg, lat_deg, L2 = True)

obs_SolElong (deg): [83.92378075] deg
lon, lat = [244.29565613] [16.68543256]


array([0.2339])

In [8]:
lon_deg

np.float64(244.29565612583932)